# Qwen2.5-VL eval — Drive edition (no browser uploads)

Apples-to-apples comparison vs Claude on the 90-image test set.

## ⚠️ Setup — do this once before running the notebook

In your Google Drive, create this structure (the adapter is **already there** from training; you only need to add the CSV):

```
MyDrive/
└── ttb_sft/
    ├── qwen2_5_vl_7b/
    │   └── adapter/                    ← already from training notebook
    │       ├── adapter_config.json
    │       ├── adapter_model.safetensors
    │       └── ...
    └── eval/
        └── cola_sample.csv             ← UPLOAD THIS ONE TIME
```

**To put `cola_sample.csv` in Drive:**
1. Open https://drive.google.com in your browser
2. Navigate to `MyDrive/ttb_sft/` (the folder is already there from training)
3. Create a new folder called `eval` inside `ttb_sft/`
4. Upload `cola_sample.csv` from `/Users/vikramsiwach/Downloads/ttb-label-verification/test/eval/data/cola_sample.csv` into that `eval/` folder

That's the only one-time setup. After this the notebook reads everything from Drive automatically — no browser upload prompts.

## How to run

1. Runtime → Change runtime type → **A100 + High-RAM** → Save
2. Runtime → Disconnect and delete runtime (clean VM)
3. Runtime → **Run all** ← 1st click. Cell 2 installs deps, auto-crashes ("session crashed" banner = expected).
4. Runtime → **Run all** ← 2nd click. Drive mount will pop a permission dialog — accept. Then everything runs end-to-end (~10 min on A100).
5. Results auto-download to `~/Downloads/qwen_outputs.jsonl`, then run `make eval-replay-qwen` locally for the side-by-side report.

## 1. Install dependencies

> Cell below installs everything then auto-restarts the kernel. After the crash banner, click **Run all again** — the cell will skip on the 2nd pass.

In [ ]:
# Direct transformers + peft path. No Unsloth (which conflicts with the
# transformers versions that have working Qwen2.5-VL). Verified deps:
#   transformers==4.51.3   — has Qwen2.5-VL + config-loading fix
#   peft>=0.13             — load the LoRA adapter
#   bitsandbytes>=0.44     — 4-bit on-the-fly quantization
#   protobuf>=5.27         — transformers 4.48+ needs this for runtime_version
#   numpy>=2.0,<2.2        — wheel ABI sweet spot
import importlib.util, subprocess, sys, os


def _have(mod): return importlib.util.find_spec(mod) is not None
def _pinned(mod, want):
    try: return __import__(mod).__version__ == want
    except Exception: return False
def _numpy_ok():
    try:
        import numpy
        p = numpy.__version__.split(".")
        return int(p[0]) == 2 and int(p[1]) < 2
    except Exception: return False


_required = ["peft", "bitsandbytes", "accelerate", "PIL"]
_ready = (_numpy_ok() and _pinned("transformers", "4.51.3")
          and all(_have(m) for m in _required))

if _ready:
    import numpy, transformers
    print(f"✓ Set up — numpy {numpy.__version__}, transformers {transformers.__version__}")
else:
    print("⏳ Installing dependencies (~3 min). Kernel will auto-restart at end.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "transformers==4.51.3", "accelerate>=0.30",
        "peft>=0.13", "bitsandbytes>=0.44",
        "qwen-vl-utils", "protobuf>=5.27", "pillow", "tqdm"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps", "numpy>=2.0,<2.2"])
    print()
    print("=" * 70)
    print("✅ Install complete. Auto-restarting kernel.")
    print("   When you see the 'session crashed' banner, click Runtime → Run all AGAIN.")
    print("=" * 70)
    os.kill(os.getpid(), 9)

## 2. Mount Drive + verify files are in place

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Drive paths (must match the setup instructions in the title cell)
DRIVE_ROOT  = Path("/content/drive/MyDrive/ttb_sft")
ADAPTER_DIR = DRIVE_ROOT / "qwen2_5_vl_7b" / "adapter"
CSV_PATH    = DRIVE_ROOT / "eval" / "cola_sample.csv"
OUTPUT_DIR  = DRIVE_ROOT / "eval"   # where we'll write the JSONL output

# Verify both required inputs exist; fail loud with explicit instructions.
if not ADAPTER_DIR.exists() or not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"\nMissing LoRA adapter at:\n  {ADAPTER_DIR}\n\n"
        f"This should already be there from the training notebook.\n"
        f"If it isn't, either:\n"
        f"  (a) re-run the training notebook to recreate it, or\n"
        f"  (b) upload the adapter folder manually to that Drive path."
    )

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"\nMissing CSV at:\n  {CSV_PATH}\n\n"
        f"Upload test/eval/data/cola_sample.csv from your Mac into\n"
        f"  MyDrive/ttb_sft/eval/\n"
        f"(create the 'eval' folder if it doesn't exist), then re-run this cell."
    )

print(f"✓ Adapter: {ADAPTER_DIR}")
print(f"  contents: {sorted(p.name for p in ADAPTER_DIR.iterdir())}")
print(f"✓ CSV:     {CSV_PATH}")
print(f"  size:    {CSV_PATH.stat().st_size:,} bytes")
print(f"✓ Output:  {OUTPUT_DIR}")

## 3. Load Qwen2.5-VL base + apply your LoRA adapter

First load downloads `Qwen/Qwen2.5-VL-7B-Instruct` from Hugging Face (~16 GB, ~5 min). After that it's cached in this Colab session.

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

print("Loading Qwen2.5-VL base (4-bit on-the-fly via bitsandbytes)...")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    quantization_config=bnb,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Applying LoRA adapter from {ADAPTER_DIR}...")
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
print("✅ Model loaded and ready for inference.")

## 4. Load CSV + download test images from the COLA Cloud CDN (~1 min)

In [ ]:
import csv, io, urllib.request, concurrent.futures
import time

rows = list(csv.DictReader(open(CSV_PATH)))
print(f"Loaded {len(rows)} rows from {CSV_PATH.name}")

IMG_DIR = Path("/content/qwen_eval_images")
IMG_DIR.mkdir(exist_ok=True)
CDN = "https://dyuie4zgfxmt6.cloudfront.net/{}.webp"

def _dl(row):
    image_id = row["ttb_image_id"]
    dest = IMG_DIR / f"{image_id}.webp"
    if dest.exists() and dest.stat().st_size > 0:
        return row, True
    try:
        req = urllib.request.Request(CDN.format(image_id),
                                     headers={"User-Agent": "ttb-eval/0.1"})
        with urllib.request.urlopen(req, timeout=20) as r:
            dest.write_bytes(r.read())
        return row, True
    except Exception as e:
        print(f"  download fail {image_id}: {e}")
        return row, False

downloaded = []
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
    for row, ok in ex.map(_dl, rows):
        if ok:
            row["_image_path"] = str(IMG_DIR / f"{row['ttb_image_id']}.webp")
            downloaded.append(row)
print(f"✓ Downloaded {len(downloaded)} / {len(rows)} images to {IMG_DIR}")

## 5. Run Qwen extraction on each image (~10 min on A100)

In [ ]:
import json, re, statistics
from PIL import Image
from collections import Counter

SYSTEM_PROMPT = (
    "You are a careful transcription assistant for U.S. TTB alcohol label review. "
    "Given ONE label panel image and the beverage type, READ what is printed and "
    "RETURN ONLY a JSON object matching the schema below. Do NOT decide compliance.\n\n"
    "If a field is not clearly visible, OMIT it from the fields object. Never guess; "
    "never substitute deposit codes (e.g. \"CA CRV\"), NOM IDs, or barcodes. Transcribe "
    "the Government Warning EXACTLY as printed (preserve case, punctuation, errors).\n\n"
    "Schema: {fields: {<field name>: {value, confidence}}, government_warning: "
    "{present, detected_text, casing_all_caps, heading_bold, body_bold, approx_font_mm, "
    "contrast_ok, separate_and_apart}, image_quality: {score, legible, note}}"
)

def _qwen_extract(image_path, beverage_type):
    """Returns (parsed_dict_or_None, raw_decoded_text)."""
    img = Image.open(image_path).convert("RGB")
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": f"Beverage type: {beverage_type}. Extract per the schema."},
        ]},
    ]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    decoded = processor.batch_decode(
        out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]
    s, e = decoded.find("{"), decoded.rfind("}")
    if s == -1 or e == -1:
        return None, decoded
    try:
        return json.loads(decoded[s:e+1]), decoded
    except json.JSONDecodeError:
        cleaned = re.sub(r",(\s*[}\]])", r"\1", decoded[s:e+1])
        try: return json.loads(cleaned), decoded
        except Exception: return None, decoded


qwen_outputs_jsonl = []
n_parsed = n_unparseable = 0
latencies = []

print(f"\nExtracting on {len(downloaded)} images...")
for i, row in enumerate(downloaded, 1):
    bev = (row.get("beverage_type") or "wine").lower()
    t0 = time.time()
    pred, raw_text = _qwen_extract(row["_image_path"], bev)
    dt = time.time() - t0
    latencies.append(dt)
    qwen_outputs_jsonl.append({
        "image_filename": row.get("image_filename") or (row["ttb_image_id"] + ".webp"),
        "ttb_image_id":   row["ttb_image_id"],
        "beverage_type":  bev,
        "raw_output":     raw_text,
        "parsed_output":  pred,
        "latency_sec":    round(dt, 3),
    })
    if pred is None: n_unparseable += 1
    else:            n_parsed += 1
    if i % 10 == 0:
        print(f"  [{i}/{len(downloaded)}] parsed={n_parsed} unparseable={n_unparseable} "
              f"latency_mean={statistics.mean(latencies):.2f}s")

print()
print("=" * 70)
print(f"Done. Parsed: {n_parsed} / Unparseable: {n_unparseable}")
print(f"Latency mean / p95: {statistics.mean(latencies):.2f}s / "
      f"{sorted(latencies)[int(len(latencies)*0.95)]:.2f}s")
print("=" * 70)

## 6. Save JSONL to Drive AND auto-download to your Mac

In [ ]:
jsonl_path_drive = OUTPUT_DIR / "qwen_outputs.jsonl"
with open(jsonl_path_drive, "w") as f:
    for rec in qwen_outputs_jsonl:
        f.write(json.dumps(rec) + "\n")
print(f"✓ Wrote {jsonl_path_drive} ({len(qwen_outputs_jsonl)} rows)")

# Also auto-download to the local Mac
from google.colab import files
files.download(str(jsonl_path_drive))

print()
print("Next steps on your Mac:")
print("  1. Move ~/Downloads/qwen_outputs.jsonl somewhere persistent")
print("  2. Run:  make eval-replay-qwen")
print("     → reads ~/Downloads/qwen_outputs.jsonl, scores it through the local")
print("       rules engine, writes test/eval/qwen_report.json, prints the")
print("       side-by-side table vs Claude's existing test/eval/report.json.")